In [2]:
import numpy as np
from scipy import stats

The goal of this code is to produce a HMM on which the viterbi algorithm can be run.

In [3]:
def EmissionProbs(observation, hidden_state):
    # This function needs to output P(observation|Hidden state)
    return 0.5 * observation * (0.9 - hidden_state)

In [4]:
class HMM:
    def __init__(self, n_components, transmat, startprob, EmissionProbs):
        self.N = n_components

        # These define the HMM
        self.transmat = transmat
        # The columns of the transition matrix need to be in the same order as the components in transmat
        # The rows of the transition matrix needs to be the opposite order of the hidden state vector

        self.pi = startprob
        self.EmissionProbs = EmissionProbs # This is a function

    def decode(self, observ):
        # observ is the observation sequence that is to be decoded to get the hidden state sequence

        viterbi = [[0] * self.N] * len(observ)
        backpointer = [0] * len(observ)

        for s in range(self.N):
            viterbi[0][s] = self.pi[s] * self.EmissionProbs(observ[0], s)
            backpointer[0] = viterbi[0].index(max(viterbi[0]))

        for t in range(1, len(observ)):
            viterbi[t] = list(np.matmul( self.transmat, viterbi[t-1] ) * np.array([self.EmissionProbs(observ[0], i) for i in range(self.N)]))

            backpointer[t] = viterbi[t].index(max(viterbi[t]))

        return backpointer


In [5]:
model = HMM(2, [[0.1, 0.9], [0.3, 0.7]], [0.2, 0.8], EmissionProbs)

In [6]:
model.decode([3,5,5,2,3])

[0, 1, 1, 0, 0]

GMM hidden layer

In [7]:
# The goal of this segment is to "train" the GMM such that we can find the mean vector and the covariance matrix required to model the data

In [8]:
# For all the feature vectors of a particular character, find their mean vector and the covariance matrix

def mean_vector(X): # The input is going to be a 2D matrix (each row is a new case)
    return np.mean(X, axis=0)

def Cov_matrix(X):
    return np.cov(X, rowvar=False)

In [9]:
X = [[2,6,3,3,2], [2,6,3,3,2], [3,6,3,2,3]]
mu = mean_vector(X)
cov = Cov_matrix(X)

In [10]:
X_test = np.random.multivariate_normal(mu, cov)
stats.multivariate_normal.pdf(X_test, mu, cov, allow_singular=True)

0.37327755602176715

In [44]:
class letter:
    def __init__(self, X): # X is the set of observations that correspond to a character (2D list)
        self.mu = mean_vector(X)
        self.cov = Cov_matrix(X)

        # The below is for the purpose of covariance update
        def multiply_with_transpose(arrays):
            res = np.array([[0] * len(X[1])] * len(X[1]))
            for array in arrays:
                array = np.array([array])
                res += np.matmul(np.transpose(array), array)
            return np.array(res)
        
        self.A = multiply_with_transpose(X)

        self.n = len(X)

    def GetProb(self, input):
        return stats.multivariate_normal.pdf(input, self.mu, self.cov, allow_singular=True)
    
    def Update(self, X): # 1D vector corresponding to 1 new case
        def Update_mean(X, Xnm1, n): # The input is one case - one 1D vector; Xnm1 - X sub(n - 1);n number of instance gone into n
            return 1 / (n+1) * ( X + (n) * Xnm1 )
        
        self.mu = Update_mean(X, self.mu, self.n)

        scov = self.n * self.cov
        scov = scov + np.matmul( ( X - self.mu ) , np.transpose( X - self.mu ) )
        self.cov = scov / (self.n + 1)

        self.n += 1



In [50]:
l = letter([[1,4,5],[3,5,8], [5,7,8], [5,0,8]])
print(l.cov)
l.Update([1,-1,8])
print(l.cov)


[[ 3.66666667 -0.66666667  2.5       ]
 [-0.66666667  8.66666667  0.        ]
 [ 2.5         0.          2.25      ]]
[[ 7.00533333  3.53866667  6.072     ]
 [ 3.53866667 11.00533333  4.072     ]
 [ 6.072       4.072       5.872     ]]


In [51]:
m = letter([[1,4,5],[3,5,8], [5,7,8], [5,0,8], [1,-1,8]])
print(l.cov)

[[ 7.00533333  3.53866667  6.072     ]
 [ 3.53866667 11.00533333  4.072     ]
 [ 6.072       4.072       5.872     ]]


CNN autoencoder

In [13]:
# The goal of this section is to convert image into feature vector

x_train = [None] # This is supposed to be the 

image_height = 64
image_width = 64

# x_train = np.reshape(x_train, (len(x_train), image_width, image_height, 1)) 

In [14]:
from tensorflow.keras import layers
from keras.layers import Input, Dense, Conv2D, MaxPooling2D, UpSampling2D, Flatten
from keras.models import Model
from keras import backend as K
import tensorflow as tf

In [18]:
input_img = Input(shape=(image_width, image_height, 1)) # Adjust channels as needed

x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Dense(8, activation='relu')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)
# This is the layer that is going to later become the feature vector: shape (None, 16, 16, 32) 

x = UpSampling2D((2, 2))(encoded)
x = Dense(8, activation='relu')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

autoencoder = Model(input_img, decoded)
autoencoder.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 64, 64, 1)]       0         
                                                                 
 conv2d_15 (Conv2D)          (None, 64, 64, 32)        320       
                                                                 
 max_pooling2d_7 (MaxPoolin  (None, 32, 32, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_16 (Conv2D)          (None, 32, 32, 32)        9248      
                                                                 
 max_pooling2d_8 (MaxPoolin  (None, 16, 16, 32)        0         
 g2D)                                                            
                                                                 
 dense_6 (Dense)             (None, 16, 16, 8)         264 

In [41]:
class Autoencoder(Model):
  def __init__(self, image_width, image_height):
    super(Autoencoder, self).__init__()

    self.encoder = tf.keras.Sequential([
      Input(shape=(image_width, image_height, 1)),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      MaxPooling2D((2, 2), padding='same'),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      MaxPooling2D((2, 2), padding='same'),
      Dense(8, activation='relu'),
      MaxPooling2D((2, 2), padding='same')
    ])
    self.decoder = tf.keras.Sequential([
      Input(shape=(int(image_width/8), int(image_height/8), 8)),
      # Input(shape=(8, 8, 8)),
      UpSampling2D((2, 2)),
      Dense(8, activation='relu'),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      UpSampling2D((2, 2)),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      UpSampling2D((2, 2)),
      Conv2D(1, (3, 3), activation='sigmoid', padding='same')
    ])

  def call(self, x):
    encoded = self.encoder(x)
    print(encoded.shape)
    decoded = self.decoder(encoded)
    return decoded


autoencoder = Autoencoder(image_height, image_width)
image = tf.random.uniform((1, image_height, image_width, 1), minval=0, maxval=1, dtype=tf.float32)
autoencoder(image)
autoencoder.summary()

(1, 8, 8, 8)
Model: "autoencoder_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_19 (Sequential)  (None, 8, 8, 8)           9832      
                                                                 
 sequential_20 (Sequential)  (None, 64, 64, 1)         11945     
                                                                 
Total params: 21777 (85.07 KB)
Trainable params: 21777 (85.07 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.fit(x_train, x_train, # X = Y here
                epochs=20,
                batch_size=256,
                shuffle=True)

In [ ]:
encoded_imgs = autoencoder.encoder(X_test).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()

# Probability generation

In [2]:
characters = """0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}"""

InitInstances = [ 0 for i in range(len(characters))]
TranProbInstances = [ [0 for i in range(len(characters))] for j in range(len(characters)) ]
# The indexing for this will work as follows [prev char index][current char index]

In [17]:
def ScanTextBlock(text, InitInstances, TranProbInstances):

    restart = True

    for c_i in range(len(text)):
        char = text[c_i]

        if char == " ":
            restart = True
        else:
            if restart:
                try:
                    InitInstances[characters.index(char)] += 1
                except ValueError:
                    pass
                restart = False
            else:
                try:
                    TranProbInstances[characters.index(text[c_i - 1])][characters.index(char)] += 1
                except ValueError:
                    pass

    return InitInstances, TranProbInstances


In [18]:
text = """Slant and slope can be introduced by both
                                                                     handwriting style and acquisition process. Ideally,
  O-line cursive script recognition (CSR) is an                     the removal results in a word image independent
important technology for applications involving                      with respect to such factors. For this reason the
automated processing of handwritten data, such as                    process is called normalization.
bank check processing and postal address reading                        The recognition techniques usually applied
(Plamondon and Srihari, 2000). Almost every CSR                      (Dynamic Programming (DP) and Hidden Mar-
system presented in the literature involves a nor-                   kov Models (HMMs)) need a fragmentation of the
malization step, which consists in removing slant                    word (Steinherz et al., 1999). In DP-based systems,
and slope. The slant is the angle between the vertical               where the isolated subunits are expected to be
direction and the direction of the strokes that, in an               characters, the normalization reduces the shape
ideal model of handwriting, are supposed to be                       variability, which simpli®es their classi®cation by
vertical (see Fig. 1). The slope is the angle between                pattern recognition techniques. In HMM-based
the horizontal direction and the direction of the line               systems, where the signal is assumed to be piece-
on which the writer aligned the word (see Fig. 1).                   wise stationary, the normalization aims to trans-
                                                                     form the handwritten data into such segments.
                                                                        In the literature, the description of the normal-"""

InitInstances, TranProbInstances = ScanTextBlock(text, InitInstances, TranProbInstances)

In [19]:
InitInstances

[0,
 6,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 57,
 39,
 21,
 12,
 6,
 8,
 0,
 10,
 39,
 0,
 2,
 6,
 2,
 12,
 12,
 16,
 0,
 20,
 49,
 68,
 2,
 4,
 22,
 0,
 0,
 0,
 2,
 0,
 2,
 2,
 0,
 6,
 0,
 4,
 8,
 0,
 0,
 0,
 4,
 0,
 2,
 2,
 0,
 0,
 11,
 6,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 14,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]